In [1]:
# ================================================================
# 1_GEN_DATA_COUNTRY.ipynb
# Build LLM-ready prompts from COUNTRY-LEVEL aggregates
# Outcome: % saying "people should try to fight global warming" (Yes)
# Stages: 1..8 per your specification
# ================================================================

import pandas as pd
import numpy as np

SRC = "data_final.csv"  # uploaded country-level file
df = pd.read_csv(SRC)

# ---- sanity checks
req = ["countrynew"]
missing = [c for c in req if c not in df.columns]
if missing:
    raise KeyError(f"❌ Missing required columns: {missing}")

print(f"✅ Loaded {SRC}: {len(df)} countries, {len(df.columns)} columns")
print("Columns:", ", ".join(df.columns))

# ================================================================
# Helpers: parse scales & formatting
# ================================================================

def as_pct(x):
    """Return percentage (0-100) as float. Accept 0–1 or 0–100."""
    if pd.isna(x): return None
    try:
        v = float(x)
    except:
        return None
    if 0 <= v <= 1:  # share
        return v * 100.0
    return v

def as_num(x, nd=1):
    if pd.isna(x): return None
    try:
        return round(float(x), nd)
    except:
        return None

def fmt_pct(x):
    v = as_pct(x)
    return f"{v:.1f}%" if v is not None else "N/A"

def fmt_num(x, nd=1):
    v = as_num(x, nd)
    return f"{v:.{nd}f}" if v is not None else "N/A"

# Support both spellings for "smaller than 1%" willingness
OWN_LESS_COL = (
    "mean_own_willingness_less"
    if "mean_own_willingness_less" in df.columns
    else ("mean_own_willigness_less" if "mean_own_willigness_less" in df.columns else None)
)

# Support either temperature column name: temp_mean OR temp_mean_2010_2019
TEMP_COL = "temp_mean" if "temp_mean" in df.columns else (
    "temp_mean_2010_2019" if "temp_mean_2010_2019" in df.columns else None
)

# ================================================================
# Stage prompt builders (Outcome: "people should try to fight GW")
# ================================================================

def build_prompt_stage1(row):
    country = row["countrynew"]
    ask = (
        f"In a nationally representative survey with a probability-based sample of approximately 1000 residents aged 15 and above in {country}, respondents were asked: "
        f"'Would you be willing to contribute 1% of your household income every month to fight global warming? This would mean that you would contribute 1 for every 100 of this income.' "
        f"Responses: Yes, No, (Don't Know), (Refused). Don't know and refused were coded as missing data. Respondents were then asked how many respondents in {country} they think are willing to contribute at least 1% of their household income every month to fight global warming. "
        f"Responses: between 0% and 100%, (Don't know), (Refused). Based on the country, estimate what respondents in {country} on average thought about how many OTHER respondents in {country} are willing to contribute at least 1% of their household income every month to fight global warming. Note: You are estimating people's BELIEFS about others' willingness, not the actual willingness itself. "
        "Respond with a single number between 0 and 100, with one decimal place."
    )
    return f"In {country}. {ask}".replace("..", ".")

def build_prompt_stage2(row):
    country = row["countrynew"]
    socio_demographics = (
        f"The average age of respondents is {fmt_num(row.get('mean_age'), 1)} years, "
        f"{fmt_pct(row.get('mean_edu'))} of the people have completed a tertiary education, "
        f"and {fmt_pct(row.get('mean_religion'))} say religion is important in daily life."
    )
    ask = (
        f"In a nationally representative survey with a probability-based sample of approximately 1000 residents aged 15 and above in {country}, respondents were asked: "
        f"'Would you be willing to contribute 1% of your household income every month to fight global warming? This would mean that you would contribute 1 for every 100 of this income.' "
        f"Responses: Yes, No, (Don't Know), (Refused). Don't know and refused were coded as missing data. Respondents were then asked how many respondents in {country} they think are willing to contribute at least 1% of their household income every month to fight global warming. "
        f"Responses: between 0% and 100%, (Don't know), (Refused). Based on the country and socio-demographic indicators, estimate what respondents in {country} on average thought about how many OTHER respondents in {country} are willing to contribute at least 1% of their household income every month to fight global warming. Note: You are estimating people's BELIEFS about others' willingness, not the actual willingness itself. "
        "Respond with a single number between 0 and 100, with one decimal place."
    )
    return f"In {country}, {socio_demographics} {ask}".replace("..", ".")

def build_prompt_stage3(row):
    country = row["countrynew"]
    macro_economic = (
        f"GDP per capita (PPP, 2021) is ${fmt_num(row.get('gdp_capita_2021'), 0)}, "
        f"the top 1% holds {fmt_pct(row.get('top1pct_income'))} of total income, "
        f"and {fmt_pct(row.get('top1pct_wealth'))} of total wealth. "
        f"The Human Development Index (2021) is {fmt_num(row.get('hdi_2021'), 3)}."
    )
    ask = (
        f"In a nationally representative survey with a probability-based sample of approximately 1000 residents aged 15 and above in {country}, respondents were asked: "
        f"'Would you be willing to contribute 1% of your household income every month to fight global warming? This would mean that you would contribute 1 for every 100 of this income.' "
        f"Responses: Yes, No, (Don't Know), (Refused). Don't know and refused were coded as missing data. Respondents were then asked how many respondents in {country} they think are willing to contribute at least 1% of their household income every month to fight global warming. "
        f"Responses: between 0% and 100%, (Don't know), (Refused). Based on the country and macro-economic indicators, estimate what respondents in {country} on average thought about how many OTHER respondents in {country} are willing to contribute at least 1% of their household income every month to fight global warming. Note: You are estimating people's BELIEFS about others' willingness, not the actual willingness itself. "
        "Respond with a single number between 0 and 100, with one decimal place."
    )
    return f"In {country}, {macro_economic} {ask}".replace("..", ".")

def build_prompt_stage4(row):
    country = row["countrynew"]
    if TEMP_COL:
        temperature = f"The average temperature from 2010 to 2019 is {fmt_num(row.get(TEMP_COL), 1)}°C."
    else:
        temperature = "Temperature data are not available."
    ask = (
        f"In a nationally representative survey with a probability-based sample of approximately 1000 residents aged 15 and above in {country}, respondents were asked: "
        f"'Would you be willing to contribute 1% of your household income every month to fight global warming? This would mean that you would contribute 1 for every 100 of this income.' "
        f"Responses: Yes, No, (Don't Know), (Refused). Don't know and refused were coded as missing data. Respondents were then asked how many respondents in {country} they think are willing to contribute at least 1% of their household income every month to fight global warming. "
        f"Responses: between 0% and 100%, (Don't know), (Refused). Based on the country and temperature data, estimate what respondents in {country} on average thought about how many OTHER respondents in {country} are willing to contribute at least 1% of their household income every month to fight global warming. Note: You are estimating people's BELIEFS about others' willingness, not the actual willingness itself. "
        "Respond with a single number between 0 and 100, with one decimal place."
    )
    return f"In {country}, {temperature} {ask}".replace("..", ".")

def build_prompt_stage5(row):
    country = row["countrynew"]
    own_main = fmt_pct(row.get("mean_own_willingness"))
    own_less = fmt_pct(row.get(OWN_LESS_COL)) if OWN_LESS_COL else "N/A"
    willingness = (
        f"In this survey, {own_main} of people said they are personally willing to contribute 1% of their income each month, "
        f"and an additional {own_less} would contribute a smaller amount."
    )
    ask = (
        f"In a nationally representative survey with a probability-based sample of approximately 1000 residents aged 15 and above in {country}, respondents were asked: "
        f"'Would you be willing to contribute 1% of your household income every month to fight global warming? This would mean that you would contribute 1 for every 100 of this income.' "
        f"Responses: Yes, No, (Don't Know), (Refused). Don't know and refused were coded as missing data. Respondents were then asked how many respondents in {country} they think are willing to contribute at least 1% of their household income every month to fight global warming. "
        f"Responses: between 0% and 100%, (Don't know), (Refused). Based on the country and the actual willingness data shown above, estimate what respondents in {country} on average thought about how many OTHER respondents in {country} are willing to contribute at least 1% of their household income every month to fight global warming. Note: You are estimating people's BELIEFS about others' willingness, not the actual willingness itself. "
        "Respond with a single number between 0 and 100, with one decimal place."
    )
    return f"In {country}, {willingness} {ask}".replace("..", ".")

def build_prompt_stage6(row):
    country = row["countrynew"]
    socio_demographics = (
        f"The average age of respondents is {fmt_num(row.get('mean_age'), 1)} years, "
        f"{fmt_pct(row.get('mean_edu'))} of the people have completed a tertiary education, "
        f"and {fmt_pct(row.get('mean_religion'))} say religion is important in daily life."
    )
    macro_economic = (
        f"GDP per capita (PPP, 2021) is ${fmt_num(row.get('gdp_capita_2021'), 0)}, "
        f"the top 1% holds {fmt_pct(row.get('top1pct_income'))} of total income, "
        f"and {fmt_pct(row.get('top1pct_wealth'))} of total wealth. "
        f"The Human Development Index (2021) is {fmt_num(row.get('hdi_2021'), 3)}."
    )
    ask = (
        f"In a nationally representative survey with a probability-based sample of approximately 1000 residents aged 15 and above in {country}, respondents were asked: "
        f"'Would you be willing to contribute 1% of your household income every month to fight global warming? This would mean that you would contribute 1 for every 100 of this income.' "
        f"Responses: Yes, No, (Don't Know), (Refused). Don't know and refused were coded as missing data. Respondents were then asked how many respondents in {country} they think are willing to contribute at least 1% of their household income every month to fight global warming. "
        f"Responses: between 0% and 100%, (Don't know), (Refused). Based on the country, socio-demographic, and macro-economic indicators, estimate what respondents in {country} on average thought about how many OTHER respondents in {country} are willing to contribute at least 1% of their household income every month to fight global warming. Note: You are estimating people's BELIEFS about others' willingness, not the actual willingness itself. "
        "Respond with a single number between 0 and 100, with one decimal place."
    )
    return f"In {country}, {socio_demographics} {macro_economic} {ask}".replace("..", ".")

def build_prompt_stage7(row):
    country = row["countrynew"]
    socio_demographics = (
        f"The average age of respondents is {fmt_num(row.get('mean_age'), 1)} years, "
        f"{fmt_pct(row.get('mean_edu'))} of the people have completed a tertiary education, "
        f"and {fmt_pct(row.get('mean_religion'))} say religion is important in daily life."
    )
    macro_economic = (
        f"GDP per capita (PPP, 2021) is ${fmt_num(row.get('gdp_capita_2021'), 0)}, "
        f"the top 1% holds {fmt_pct(row.get('top1pct_income'))} of total income, "
        f"and {fmt_pct(row.get('top1pct_wealth'))} of total wealth. "
        f"The Human Development Index (2021) is {fmt_num(row.get('hdi_2021'), 3)}."
    )
    if TEMP_COL:
        temperature = f"The average temperature from 2010 to 2019 is {fmt_num(row.get(TEMP_COL), 1)}°C."
    else:
        temperature = "Temperature data are not available."
    ask = (
        f"In a nationally representative survey with a probability-based sample of approximately 1000 residents aged 15 and above in {country}, respondents were asked: "
        f"'Would you be willing to contribute 1% of your household income every month to fight global warming? This would mean that you would contribute 1 for every 100 of this income.' "
        f"Responses: Yes, No, (Don't Know), (Refused). Don't know and refused were coded as missing data. Respondents were then asked how many respondents in {country} they think are willing to contribute at least 1% of their household income every month to fight global warming. "
        f"Responses: between 0% and 100%, (Don't know), (Refused). Based on the country, socio-demographic, macro-economic indicators, and temperature data, estimate what respondents in {country} on average thought about how many OTHER respondents in {country} are willing to contribute at least 1% of their household income every month to fight global warming. Note: You are estimating people's BELIEFS about others' willingness, not the actual willingness itself. "
        "Respond with a single number between 0 and 100, with one decimal place."
    )
    return f"In {country}, {socio_demographics} {macro_economic} {temperature} {ask}".replace("..", ".")

def build_prompt_stage8(row):
    country = row["countrynew"]
    socio_demographics = (
        f"The average age of respondents is {fmt_num(row.get('mean_age'), 1)} years, "
        f"{fmt_pct(row.get('mean_edu'))} of the people have completed a tertiary education, "
        f"and {fmt_pct(row.get('mean_religion'))} say religion is important in daily life."
    )
    macro_economic = (
        f"GDP per capita (PPP, 2021) is ${fmt_num(row.get('gdp_capita_2021'), 0)}, "
        f"the top 1% holds {fmt_pct(row.get('top1pct_income'))} of total income, "
        f"and {fmt_pct(row.get('top1pct_wealth'))} of total wealth. "
        f"The Human Development Index (2021) is {fmt_num(row.get('hdi_2021'), 3)}."
    )
    if TEMP_COL:
        temperature = f"The average temperature from 2010 to 2019 is {fmt_num(row.get(TEMP_COL), 1)}°C."
    else:
        temperature = "Temperature data are not available."
    own_main = fmt_pct(row.get("mean_own_willingness"))
    own_less = fmt_pct(row.get(OWN_LESS_COL)) if OWN_LESS_COL else "N/A"
    willingness = (
        f"In this survey, {own_main} of people said they are personally willing to contribute 1% of their income each month, "
        f"and an additional {own_less} would contribute a smaller amount."
    )
    ask = (
        f"In a nationally representative survey with a probability-based sample of approximately 1000 residents aged 15 and above in {country}, respondents were asked: "
        f"'Would you be willing to contribute 1% of your household income every month to fight global warming? This would mean that you would contribute 1 for every 100 of this income.' "
        f"Responses: Yes, No, (Don't Know), (Refused). Don't know and refused were coded as missing data. Respondents were then asked how many respondents in {country} they think are willing to contribute at least 1% of their household income every month to fight global warming. "
        f"Responses: between 0% and 100%, (Don't know), (Refused). Based on the country, socio-demographic, macro-economic indicators, temperature data, and the actual willingness data shown above, estimate what respondents in {country} on average thought about how many OTHER respondents in {country} are willing to contribute at least 1% of their household income every month to fight global warming. Note: You are estimating people's BELIEFS about others' willingness, not the actual willingness itself. "
        "Respond with a single number between 0 and 100, with one decimal place."
    )
    return f"In {country}, {socio_demographics} {macro_economic} {temperature} {willingness} {ask}".replace("..", ".")

# Quick smoke tests (optional — comment out if you prefer a silent run)
print("\n=== SMOKE TESTS ===")
print("\nStage 1:")
print(build_prompt_stage1({"countrynew": "Afghanistan"}))
print("\nStage 2:")
print(build_prompt_stage2({"countrynew": "Afghanistan", "mean_age": 32.1, "mean_edu": 2.5, "mean_religion": 0.8}))
print("\nStage 3:")
print(build_prompt_stage3({"countrynew": "Afghanistan", "gdp_capita_2021": 3829.66, "top1pct_income": 0.1469, "top1pct_wealth": 0.2419, "hdi_2021": 0.478}))
print("\nStage 4:")
test_temp_col = TEMP_COL if TEMP_COL else "temp_mean_2010_2019"
print(build_prompt_stage4({"countrynew": "Afghanistan", test_temp_col: 15.2}))
print("\nStage 5:")
test_own_less = OWN_LESS_COL if OWN_LESS_COL else "mean_own_willigness_less"
print(build_prompt_stage5({"countrynew": "Afghanistan", "mean_own_willingness": 0.82, test_own_less: 0.24}))
print("\nStage 6:")
print(build_prompt_stage6({"countrynew": "Afghanistan", "mean_age": 32.1, "mean_edu": 2.5, "mean_religion": 0.8, "gdp_capita_2021": 3829.66, "top1pct_income": 0.1469, "top1pct_wealth": 0.2419, "hdi_2021": 0.478}))
print("\nStage 7:")
print(build_prompt_stage7({"countrynew": "Afghanistan", "mean_age": 32.1, "mean_edu": 2.5, "mean_religion": 0.8, "gdp_capita_2021": 3829.66, "top1pct_income": 0.1469, "top1pct_wealth": 0.2419, "hdi_2021": 0.478, test_temp_col: 15.2}))
print("\nStage 8:")
print(build_prompt_stage8({"countrynew": "Afghanistan", "mean_age": 32.1, "mean_edu": 2.5, "mean_religion": 0.8, "gdp_capita_2021": 3829.66, "top1pct_income": 0.1469, "top1pct_wealth": 0.2419, "hdi_2021": 0.478, test_temp_col: 15.2, "mean_own_willingness": 0.82, test_own_less: 0.24}))

# ================================================================
# Build prompt columns for all countries
# ================================================================

df["prompt_stage1"] = df.apply(build_prompt_stage1, axis=1)
df["prompt_stage2"] = df.apply(build_prompt_stage2, axis=1)
df["prompt_stage3"] = df.apply(build_prompt_stage3, axis=1)
df["prompt_stage4"] = df.apply(build_prompt_stage4, axis=1)
df["prompt_stage5"] = df.apply(build_prompt_stage5, axis=1)
df["prompt_stage6"] = df.apply(build_prompt_stage6, axis=1)
df["prompt_stage7"] = df.apply(build_prompt_stage7, axis=1)
df["prompt_stage8"] = df.apply(build_prompt_stage8, axis=1)

# ================================================================
# Optional: compute TRUE PI gap if you have the columns
# (kept here for continuity; not needed for this specific outcome)
# ================================================================

if ("mean_other_willingness" in df.columns) and ("mean_own_willingness" in df.columns):
    own = df["mean_own_willingness"].apply(as_pct) / 100.0
    oth = df["mean_other_willingness"].apply(as_pct) / 100.0
    df["pi_gap_true"] = own - oth

# ================================================================
# Save LLM-ready prompts
# ================================================================

cols = ["countrynew"] + [f"prompt_stage{i}" for i in range(1,9)]
# Keep useful context columns (optional)
for opt in [
    "mean_own_willingness", OWN_LESS_COL, "mean_other_willingness", "pi_gap_true",
    "hdi_2021", "gdp_capita_2021", "top1pct_income", "top1pct_wealth",
    "mean_age", "mean_edu", "mean_religion", TEMP_COL
]:
    if isinstance(opt, str) and opt in df.columns:
        cols.append(opt)

out = df[cols].copy()
out.to_csv("country_llm_prompts_outcome2_8stages.csv", index=False, encoding="utf-8-sig")
print("\n✅ Exported country_llm_prompts_outcome2_8stages.csv")
print(f"   Contains {len(out)} countries × {len(cols)} columns")
out.head(3)

✅ Loaded data_final.csv: 125 countries, 17 columns
Columns: countrynew, mean_age, mean_gender, mean_edu, mean_religion, mean_own_willingness, mean_own_willigness_less, mean_other_willingness, wtc_own, wtc_other, hdi_2021, gdp_capita_2021, top1pct_income, top1pct_wealth, temp_mean_2010_2019, other_fight_cc, govt_fight_cc

=== SMOKE TESTS ===

Stage 1:
In Afghanistan. In a nationally representative survey with a probability-based sample of approximately 1000 residents aged 15 and above in Afghanistan, respondents were asked: 'Would you be willing to contribute 1% of your household income every month to fight global warming? This would mean that you would contribute 1 for every 100 of this income.' Responses: Yes, No, (Don't Know), (Refused). Don't know and refused were coded as missing data. Respondents were then asked how many respondents in Afghanistan they think are willing to contribute at least 1% of their household income every month to fight global warming. Responses: between 0% a

,countrynew,prompt_stage1,prompt_stage2,prompt_stage3,prompt_stage4,prompt_stage5,prompt_stage6,prompt_stage7,prompt_stage8,mean_own_willingness,...,mean_other_willingness,pi_gap_true,hdi_2021,gdp_capita_2021,top1pct_income,top1pct_wealth,mean_age,mean_edu,mean_religion,temp_mean_2010_2019
0,Afghanistan,In Afghanistan. In a nationally representative...,"In Afghanistan, The average age of respondents...","In Afghanistan, GDP per capita (PPP, 2021) is ...","In Afghanistan, The average temperature from 2...","In Afghanistan, In this survey, 82.0% of peopl...","In Afghanistan, The average age of respondents...","In Afghanistan, The average age of respondents...","In Afghanistan, The average age of respondents...",0.820461,...,0.405198,0.415263,0.478,3829.6620,0.1469,0.2419,32.115964,0.034356,0.976127,13.69
1,Albania,In Albania. In a nationally representative sur...,"In Albania, The average age of respondents is ...","In Albania, GDP per capita (PPP, 2021) is $158...","In Albania, The average temperature from 2010 ...","In Albania, In this survey, 71.3% of people sa...","In Albania, The average age of respondents is ...","In Albania, The average age of respondents is ...","In Albania, The average age of respondents is ...",0.712788,...,0.440810,0.271978,0.796,15888.9033,0.0886,0.2306,42.739299,0.122144,0.576923,12.93
2,Algeria,In Algeria. In a nationally representative sur...,"In Algeria, The average age of respondents is ...","In Algeria, GDP per capita (PPP, 2021) is $158...","In Algeria, The average temperature from 2010 ...","In Algeria, In this survey, 54.6% of people sa...","In Algeria, The average age of respondents is ...","In Algeria, The average age of respondents is ...","In Algeria, The average age of respondents is ...",0.545846,...,0.406771,0.139075,0.745,15827.7456,0.2264,0.2802,37.420958,0.108559,NaN,12.93


<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=feb9f195-de2a-416f-b8f1-09efca4e954f' target="_blank">
 </img>
Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>